<a href="https://colab.research.google.com/github/RohanDevre36/Minimum-wage-causal-analysis/blob/main/01_Data_Collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Minimum_Wage_Causal_Analysis"
)

folders = ["data_raw", "data_processed", "notebooks", "outputs"]

for folder in folders:
    path = PROJECT_DIR / folder
    status = "Found" if path.is_dir() else "Missing"
    print(f"{folder}: {status}")

data_raw: Found
data_processed: Found
notebooks: Found
outputs: Found


# Causal Impact of State Minimum Wage Increases
## Employment and Earnings in the United States

### Research question
How do state minimum wage increases affect employment and average
weekly wages in food services and retail?

### Project objective
Estimate how these outcomes changed after a minimum wage increase,
compared with what would likely have happened without the policy.

### Planned approach
- Collect and combine public government datasets.
- Explore trends and assess suitable comparison states.
- Apply Difference-in-Differences and an event study.
- Evaluate assumptions, uncertainty, and sensitivity to modeling choices.
- Communicate findings and policy implications.

### Scope
The policy episode, comparison states, and study period will be
selected after checking policy histories and data availability.

### Project status
Project folders created and Google Drive connection verified.
Data collection has not yet started.

In [6]:
import sys
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy
import statsmodels
import requests

packages = {
    "Python": sys.version.split()[0],
    "pandas": pd.__version__,
    "NumPy": np.__version__,
    "Matplotlib": matplotlib.__version__,
    "Seaborn": sns.__version__,
    "SciPy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
    "requests": requests.__version__,
}

for name, version in packages.items():
    print(f"{name}: {version}")

print("\nPackage check completed successfully.")

Python: 3.13.16
pandas: 2.2.3
NumPy: 2.1.3
Matplotlib: 3.10.0
Seaborn: 0.13.2
SciPy: 1.16.3
statsmodels: 0.15.0
requests: 2.32.4

Package check completed successfully.


In [7]:
from datetime import datetime, timezone

# Record the packages used in this notebook.
requirements = "\n".join(
    f"{name}=={version}"
    for name, version in packages.items()
    if name != "Python"
)

requirements_path = PROJECT_DIR / "requirements.txt"
requirements_path.write_text(requirements + "\n", encoding="utf-8")

# Record Python version and the date separately.
environment_path = PROJECT_DIR / "environment.txt"
environment_path.write_text(
    f"Python: {packages['Python']}\n"
    f"Recorded (UTC): {datetime.now(timezone.utc).isoformat()}\n",
    encoding="utf-8"
)

print("Saved requirements.txt and environment.txt")
print(f"Project folder: {PROJECT_DIR}")

Saved requirements.txt and environment.txt
Project folder: /content/drive/MyDrive/Minimum_Wage_Causal_Analysis


## Data Collection Plan

We will combine policy information with state-level labor market data.

| Source | Data needed | Purpose |
|---|---|---|
| US Department of Labor and state labor agencies | Minimum wage rates and effective dates | Identify policy changes |
| BLS Quarterly Census of Employment and Wages (QCEW) | Employment and average weekly wages in food services and retail | Measure industry outcomes |
| BLS Local Area Unemployment Statistics (LAUS) | State unemployment rates | Describe broader labor market conditions |

### Planned data structure
Each row will represent one state, industry, and time period.
We will choose the time frequency after checking data availability.

### Important measurement distinction
Average weekly wages are not hourly wage rates. They can change
because of hours worked or changes in the composition of employees,
as well as changes in pay.

### Study selection
We will verify policy dates and data coverage before selecting the
treated states, comparison states, and study period. Comparison
selection will use policy histories and pre-policy information,
rather than the size or significance of estimated effects.

In [8]:
from io import BytesIO

url = "https://data.bls.gov/cew/data/api/2019/1/industry/722.csv"

response = requests.get(url, timeout=60)
response.raise_for_status()

sample = pd.read_csv(
    BytesIO(response.content),
    dtype={"area_fips": str, "industry_code": str}
)

# Check the response before saving it.
required_columns = {
    "area_fips", "industry_code", "own_code",
    "month1_emplvl", "avg_wkly_wage"
}

if not required_columns.issubset(sample.columns):
    raise ValueError("The download did not contain the expected BLS columns.")

raw_path = PROJECT_DIR / "data_raw" / "qcew_2019_q1_722.csv"
raw_path.write_bytes(response.content)

print(f"Downloaded {sample.shape[0]:,} rows and {sample.shape[1]} columns.")
print(f"Saved: {raw_path.name}")

display(sample.head())

Downloaded 4,152 rows and 42 columns.
Saved: qcew_2019_q1_722.csv


,area_fips,own_code,industry_code,agglvl_code,size_code,year,qtr,disclosure_code,qtrly_estabs,month1_emplvl,...,oty_month3_emplvl_chg,oty_month3_emplvl_pct_chg,oty_total_qtrly_wages_chg,oty_total_qtrly_wages_pct_chg,oty_taxable_qtrly_wages_chg,oty_taxable_qtrly_wages_pct_chg,oty_qtrly_contributions_chg,oty_qtrly_contributions_pct_chg,oty_avg_wkly_wage_chg,oty_avg_wkly_wage_pct_chg
0,01000,1,722,55,0,2019,1,NaN,6,208,...,-12,-5.5,1614,0.1,0,0.0,0,0.0,24,6.1
1,01000,3,722,55,0,2019,1,N,145,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,01000,5,722,55,0,2019,1,NaN,8642,163474,...,2269,1.4,30898280,4.9,22526342,4.0,25904,0.5,9,3.0
3,01001,3,722,75,0,2019,1,N,1,0,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
4,01001,5,722,75,0,2019,1,NaN,67,1195,...,-18,-1.5,88596,2.1,56550,1.5,-3632,-11.2,12,4.6


In [9]:
print('total rows:', sample.shape[0])
print('total columns:', sample.shape[1])

total rows: 4152
total columns: 42


In [10]:
# Before filtering, will check how BLS identifies the records.

print("ownership codes and record counts:")
print(sample["own_code"].value_counts().sort_index())

print("\nFirst 15 geographic codes:")
print(sample["area_fips"].drop_duplicates().head(15).tolist())

print("\nAvailable columns:")
print(sample.columns.tolist())



ownership codes and record counts:
own_code
1     166
2       7
3     287
5    3692
Name: count, dtype: int64

First 15 geographic codes:
['01000', '01001', '01003', '01005', '01007', '01009', '01011', '01013', '01015', '01017', '01019', '01021', '01023', '01025', '01027']

Available columns:
['area_fips', 'own_code', 'industry_code', 'agglvl_code', 'size_code', 'year', 'qtr', 'disclosure_code', 'qtrly_estabs', 'month1_emplvl', 'month2_emplvl', 'month3_emplvl', 'total_qtrly_wages', 'taxable_qtrly_wages', 'qtrly_contributions', 'avg_wkly_wage', 'lq_disclosure_code', 'lq_qtrly_estabs', 'lq_month1_emplvl', 'lq_month2_emplvl', 'lq_month3_emplvl', 'lq_total_qtrly_wages', 'lq_taxable_qtrly_wages', 'lq_qtrly_contributions', 'lq_avg_wkly_wage', 'oty_disclosure_code', 'oty_qtrly_estabs_chg', 'oty_qtrly_estabs_pct_chg', 'oty_month1_emplvl_chg', 'oty_month1_emplvl_pct_chg', 'oty_month2_emplvl_chg', 'oty_month2_emplvl_pct_chg', 'oty_month3_emplvl_chg', 'oty_month3_emplvl_pct_chg', 'oty_total_qtrly

In [14]:
# FIPS codes identify geographic areas.
# This list inclides the 50 states and washington, DC
# Keep them as strings so leading zeros, such as "01", are preserved.

state_fips = """
01 02 04 05 06
08 09 10 11 12 13
15 16 17 18 19 20 21 22
23 24 25 26 27 28 29 30
31 32 33 34 35 36 37 38
39 40 41 42 44 45 46 47
48 49 50 51 53 54 55 56
""".split()

# Convert each two-digit code into a five charater statewide code.
#Example: alabama "01" becomes "01000".
state_area_codes = [code + "000" for code in state_fips]

#select onlu the records that meet all four conditions below.
#The & symbol menas "AND": .eq() menas "equals".
state_sample = sample.loc[
    #keep only the 50 states and washington DC
    sample["area_fips"].isin(state_area_codes)
    #owenership code 5 menas private-sector emplyoers
    & sample["own_code"].eq(5)
    # Aggregation code 55 means statewide, three-digit industry records.
    & sample["agglvl_code"].eq(55)
    # Industry 722 means food services and drinking places.
    & sample["industry_code"].eq("722")

# Make an independent copy so we can work without changing sample.
].copy()

#Check how many records remain. we expect one per state/DC
print("State/DC records:", len(state_sample))

#check whether any geographic area appears more than once
# # Zero duplicates is the expected result for this single-quarter sample.
print(
    "Duplicate areas:",
    state_sample["area_fips"].duplicated().sum()
)

# Inspect disclosure flags before using employment or wage values.
# Some data may be withheld to protect employer confidentiality.
# Replace missing flags with "(blank)" only for this printed summary.
print("\nDisclosure flags:")
print(
    state_sample["disclosure_code"]
    .fillna("(blank)")
    .value_counts()
)

# Preview the geographic code, time period, monthly employment,
# and average weekly wage for the first five selected records.
display(
    state_sample[
        [
            "area_fips",
            "year",
            "qtr",
            "month1_emplvl",
            "month2_emplvl",
            "month3_emplvl",
            "avg_wkly_wage"
        ]
    ].head()
)

State/DC records: 51
Duplicate areas: 0

Disclosure flags:
disclosure_code
(blank)    51
Name: count, dtype: int64


,area_fips,year,qtr,month1_emplvl,month2_emplvl,month3_emplvl,avg_wkly_wage
2,01000,2019,1,163474,166391,168111,309
99,02000,2019,1,20023,20303,20491,416
131,04000,2019,1,236992,240813,244560,409
155,05000,2019,1,94649,95886,97053,307
235,06000,2019,1,1423433,1441824,1456022,451


In [15]:
##Calculate average monthly employment for the quarter
# These columns contain employment counts for each month of Q1.
employment_columns = [
    "month1_emplvl",
    "month2_emplvl",
    "month3_emplvl"
]

# Calculate average monthly employment during the quarter.
# skipna=False prevents calculating a partial average if a month is missing.
state_sample["avg_monthly_employment"] = (
    state_sample[employment_columns].mean(axis=1, skipna=False)
)

# Inspect missing values in the two outcomes we plan to analyze.
outcome_columns = ["avg_monthly_employment", "avg_wkly_wage"]

print("Missing values:")
print(state_sample[outcome_columns].isna().sum())

# Show the first five records with a fresh, consecutive display index.
display(
    state_sample[
        ["area_fips", "year", "qtr",
         "avg_monthly_employment", "avg_wkly_wage"]
    ].reset_index(drop=True).head()
)

Missing values:
avg_monthly_employment    0
avg_wkly_wage             0
dtype: int64


,area_fips,year,qtr,avg_monthly_employment,avg_wkly_wage
0,01000,2019,1,1.659920e+05,309
1,02000,2019,1,2.027233e+04,416
2,04000,2019,1,2.407883e+05,409
3,05000,2019,1,9.586267e+04,307
4,06000,2019,1,1.440426e+06,451


In [16]:
##Save the prepared sample
# Keep identifiers, disclosure information, and our two outcomes.
# Retain industry and ownership codes so the file describes its scope.
columns_to_save = [
    "area_fips",
    "industry_code",
    "own_code",
    "year",
    "qtr",
    "disclosure_code",
    "avg_monthly_employment",
    "avg_wkly_wage"
]

# Sort by geographic code and create consecutive row indexes.
prepared_sample = (
    state_sample[columns_to_save]
    .sort_values("area_fips")
    .reset_index(drop=True)
)

# Save the prepared data separately from the original raw download.
output_path = (
    PROJECT_DIR / "data_processed"
    / "qcew_state_private_food_services_2019_q1.csv"
)

# index=False prevents writing the DataFrame's row index into the CSV.
prepared_sample.to_csv(output_path, index=False)

print(f"Saved: {output_path.name}")
print(f"Rows: {len(prepared_sample)}")

# Format the preview for readability without changing stored values.
display(
    prepared_sample.head().style.format({
        "avg_monthly_employment": "{:,.2f}",
        "avg_wkly_wage": "${:,.0f}"
    })
)

Saved: qcew_state_private_food_services_2019_q1.csv
Rows: 51


,area_fips,industry_code,own_code,year,qtr,disclosure_code,avg_monthly_employment,avg_wkly_wage
0,01000,722,5,2019,1,nan,"165,992.00",$309
1,02000,722,5,2019,1,nan,"20,272.33",$416
2,04000,722,5,2019,1,nan,"240,788.33",$409
3,05000,722,5,2019,1,nan,"95,862.67",$307
4,06000,722,5,2019,1,nan,"1,440,426.33",$451


###Define our proposed policy case

## Proposed Policy Case: New Jersey, January 2014

### Research question
How did New Jersey's minimum wage policy beginning in January 2014
affect private-sector food services employment and average weekly wages?

### Proposed study window
- Before the policy: 2010–2013.
- After the policy: 2014–2018.
- Frequency: quarterly.

This window ends before New Jersey's separate 2019 wage expansion
and the COVID-19 pandemic.

### Treatment definition
New Jersey is the treated state. The intervention includes the
January 2014 increase and subsequent inflation-linked adjustments.

### Comparison states
Not selected yet. We will review their minimum wage histories,
pre-policy outcome trends, and economic comparability.

### Issues to investigate
- Employers may have responded before the effective date.
- Other state or local policies may affect the same outcomes.
- Tipped workers have different wage rules.
- Cross-state commuting and business activity may create spillovers.
- A single treated state requires careful statistical inference.

### Decision rule
This case is provisional. We will assess data coverage and comparison
quality before treating it as a credible causal study.

### Policy sources
- https://fred.stlouisfed.org/data/STTMINWGNJ
- https://www.nj.gov/labor/forms_pdfs/lwdhome/MinWage2015.pdf

In [19]:
##Check data availability at the start of our study
from io import BytesIO

# Test the earliest quarter in our proposed study window.
test_year = 2010
test_quarter = 1

# Industry 722 represents food services and drinking places.
url = (
    f"https://data.bls.gov/cew/data/api/"
    f"{test_year}/{test_quarter}/industry/722.csv"
)

# Request the CSV. Stop with an error if the download fails.
response = requests.get(url, timeout=60)
response.raise_for_status()

# Preserve geographic and industry codes as strings.
test_data = pd.read_csv(
    BytesIO(response.content),
    dtype={"area_fips": str, "industry_code": str}
)

# Confirm that the response contains the columns we need.
required_columns = {
    "area_fips", "own_code", "agglvl_code", "industry_code",
    "year", "qtr", "disclosure_code",
    "month1_emplvl", "month2_emplvl", "month3_emplvl",
    "avg_wkly_wage"
}

if not required_columns.issubset(test_data.columns):
    raise ValueError("The download is missing required BLS columns.")

# Save the original download without modifying its contents.
raw_path = PROJECT_DIR / "data_raw" / "qcew_2010_q1_722.csv"
raw_path.write_bytes(response.content)

# Apply the same state/DC and private-sector filters as before.
test_states = test_data.loc[
    test_data["area_fips"].isin(state_area_codes)
    & test_data["own_code"].eq(5)
    & test_data["agglvl_code"].eq(55)
    & test_data["industry_code"].eq("722")
].copy()

# Check geographic coverage and duplicate records.
print("State/DC records:", len(test_states))
print("Duplicate areas:", test_states["area_fips"].duplicated().sum())

# Inspect whether any state records carry disclosure flags.
print("\nDisclosure flags:")
print(
    test_states["disclosure_code"]
    .fillna("(blank)")
    .value_counts()
)

# New Jersey's statewide geographic code is 34000.
print("\nNew Jersey record:")
display(
    test_states.loc[
        test_states["area_fips"].eq("34000"),
        [
            "area_fips", "year", "qtr",
            "month1_emplvl", "month2_emplvl",
            "month3_emplvl", "avg_wkly_wage"
        ]
    ]
)

HTTPError: 404 Client Error: Not Found for url: https://data.bls.gov/cew/data/api/2010/1/industry/722.csv

In [18]:
# Test the historical archive route for 2010.
archive_url = (
    "https://data.bls.gov/cew/data/files/"
    "2010/csv/2010_qtrly_by_industry.zip"
)

# stream=True checks the response without downloading the entire ZIP.
# The archive contains many industries and may be large.
with requests.get(
    archive_url,
    stream=True,
    timeout=60
) as archive_response:

    print("HTTP status:", archive_response.status_code)
    print(
        "Content type:",
        archive_response.headers.get("Content-Type", "Not provided")
    )

HTTP status: 200
Content type: application/zip


In [20]:
#Download the 2010 archive
from zipfile import ZipFile, is_zipfile

# Save the archive in our raw-data folder.
archive_path = PROJECT_DIR / "data_raw" / "2010_qtrly_by_industry.zip"

# Reuse an existing valid ZIP to avoid downloading it again.
if archive_path.exists() and is_zipfile(archive_path):
    print("Using the archive already saved.")
else:
    # Download into a temporary file first.
    # An interrupted download will not look like a completed archive.
    temporary_path = archive_path.with_suffix(".zip.part")

    print("Downloading the 2010 archive. This may take a few minutes.")

    with requests.get(
        archive_url, stream=True, timeout=(30, 120)
    ) as download_response:
        download_response.raise_for_status()

        with temporary_path.open("wb") as file:
            # Write the download in chunks instead of loading it all into RAM.
            for chunk in download_response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)

    # Confirm that the downloaded file is recognizable as a ZIP.
    if not is_zipfile(temporary_path):
        raise ValueError("The downloaded file is not a valid ZIP archive.")

    # Rename only after the download finishes successfully.
    temporary_path.replace(archive_path)
    print("Download completed.")

# Inspect filenames without extracting the entire archive.
with ZipFile(archive_path) as archive:
    archive_members = archive.namelist()

print(f"\nFiles in archive: {len(archive_members):,}")
print("\nFirst 10 filenames:")

for filename in archive_members[:10]:
    print(filename)

Download completed.

Files in archive: 2,364

First 10 filenames:
2010.q1-q4.by_industry/
2010.q1-q4.by_industry/2010.q1-q4 10 Total, all industries.csv
2010.q1-q4.by_industry/2010.q1-q4 101 Goods-producing.csv
2010.q1-q4.by_industry/2010.q1-q4 1011 Natural resources and mining.csv
2010.q1-q4.by_industry/2010.q1-q4 1012 Construction.csv
2010.q1-q4.by_industry/2010.q1-q4 1013 Manufacturing.csv
2010.q1-q4.by_industry/2010.q1-q4 102 Service-providing.csv
2010.q1-q4.by_industry/2010.q1-q4 1021 Trade, transportation, and utilities.csv
2010.q1-q4.by_industry/2010.q1-q4 1022 Information.csv
2010.q1-q4.by_industry/2010.q1-q4 1023 Financial activities.csv


In [21]:
##Find the food services file
# Find the CSV for industry 722: food services and drinking places.
# Spaces around "722" prevent matching narrower codes such as 7221.
food_service_files = [
    filename
    for filename in archive_members
    if " 722 " in filename and filename.endswith(".csv")
]

print("Matching files:", len(food_service_files))

for filename in food_service_files:
    print(filename)

# Stop if the archive does not contain exactly one matching file.
if len(food_service_files) != 1:
    raise ValueError("Expected exactly one industry 722 file.")

# Read only the selected CSV directly from the ZIP.
# No need to extract the other files.
with ZipFile(archive_path) as archive:
    with archive.open(food_service_files[0]) as csv_file:
        food_2010 = pd.read_csv(
            csv_file,
            dtype={"area_fips": str, "industry_code": str}
        )

# Inspect the historical file before applying our filters.
print("\nDataset shape:", food_2010.shape)
print("\nColumns:")
print(food_2010.columns.tolist())

display(food_2010.head())

Matching files: 1
2010.q1-q4.by_industry/2010.q1-q4 722 Food services and drinking places.csv

Dataset shape: (16560, 47)

Columns:
['area_fips', 'own_code', 'industry_code', 'agglvl_code', 'size_code', 'year', 'qtr', 'disclosure_code', 'area_title', 'own_title', 'industry_title', 'agglvl_title', 'size_title', 'qtrly_estabs_count', 'month1_emplvl', 'month2_emplvl', 'month3_emplvl', 'total_qtrly_wages', 'taxable_qtrly_wages', 'qtrly_contributions', 'avg_wkly_wage', 'lq_disclosure_code', 'lq_qtrly_estabs_count', 'lq_month1_emplvl', 'lq_month2_emplvl', 'lq_month3_emplvl', 'lq_total_qtrly_wages', 'lq_taxable_qtrly_wages', 'lq_qtrly_contributions', 'lq_avg_wkly_wage', 'oty_disclosure_code', 'oty_qtrly_estabs_count_chg', 'oty_qtrly_estabs_count_pct_chg', 'oty_month1_emplvl_chg', 'oty_month1_emplvl_pct', 'oty_month2_emplvl_chg', 'oty_month2_emplvl_pct', 'oty_month3_emplvl_chg', 'oty_month3_emplvl_pct', 'oty_total_qtrly_wages_chg', 'oty_total_qtrly_wages_pct', 'oty_taxable_qtrly_wages_chg', 'o

,area_fips,own_code,industry_code,agglvl_code,size_code,year,qtr,disclosure_code,area_title,own_title,...,oty_month3_emplvl_chg,oty_month3_emplvl_pct,oty_total_qtrly_wages_chg,oty_total_qtrly_wages_pct,oty_taxable_qtrly_wages_chg,oty_taxable_qtrly_wages_chg.1,oty_qtrly_contributions_chg,oty_qtrly_contributions_pct,oty_avg_wkly_wage_chg,oty_avg_wkly_wage_pct
0,01000,1,722,55,0,2010,1,NaN,Alabama -- Statewide,Federal Government,...,70,45.8,209356,27.9,0,0.0,0,0.0,-42,-11.1
1,01000,1,722,55,0,2010,2,NaN,Alabama -- Statewide,Federal Government,...,65,40.9,217940,31.0,0,0.0,0,0.0,-20,-5.9
2,01000,1,722,55,0,2010,3,NaN,Alabama -- Statewide,Federal Government,...,55,28.9,230867,28.5,0,0.0,0,0.0,-17,-4.8
3,01000,1,722,55,0,2010,4,NaN,Alabama -- Statewide,Federal Government,...,61,34.3,292380,35.2,0,0.0,0,0.0,16,4.7
4,01000,3,722,55,0,2010,1,N,Alabama -- Statewide,Local Government,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0


In [22]:
##Filter and check all four quarters of 2010

# Keep private-sector food services records for the 50 states and DC.
# Unlike our earlier sample, this file contains four quarters.
states_2010 = food_2010.loc[
    food_2010["area_fips"].isin(state_area_codes)
    & food_2010["own_code"].eq(5)
    & food_2010["agglvl_code"].eq(55)
    & food_2010["industry_code"].eq("722")
    & food_2010["year"].eq(2010)
    & food_2010["qtr"].isin([1, 2, 3, 4])
].copy()

# Each state should appear once PER QUARTER.
# Therefore, check duplicates using geography, year, and quarter together.
record_key = ["area_fips", "year", "qtr"]

print("Total records:", len(states_2010))
print("Duplicate state-quarter records:",
      states_2010.duplicated(subset=record_key).sum())

# Expect 51 state/DC records in each quarter: 204 records overall.
print("\nRecords per quarter:")
print(states_2010.groupby("qtr").size())

# Inspect disclosure flags before calculating or interpreting outcomes.
print("\nDisclosure flags:")
print(
    states_2010["disclosure_code"]
    .fillna("(blank)")
    .value_counts()
)

# Check whether employment or wage values are missing.
outcome_fields = [
    "month1_emplvl", "month2_emplvl",
    "month3_emplvl", "avg_wkly_wage"
]

print("\nMissing values:")
print(states_2010[outcome_fields].isna().sum())

# Display New Jersey's four quarterly records.
display(
    states_2010.loc[
        states_2010["area_fips"].eq("34000"),
        ["area_title", "year", "qtr"] + outcome_fields
    ].sort_values("qtr")
)


Total records: 204
Duplicate state-quarter records: 0

Records per quarter:
qtr
1    51
2    51
3    51
4    51
dtype: int64

Disclosure flags:
disclosure_code
(blank)    204
Name: count, dtype: int64

Missing values:
month1_emplvl    0
month2_emplvl    0
month3_emplvl    0
avg_wkly_wage    0
dtype: int64


,area_title,year,qtr,month1_emplvl,month2_emplvl,month3_emplvl,avg_wkly_wage
8288,New Jersey -- Statewide,2010,1,207298,206115,210047,313
8289,New Jersey -- Statewide,2010,2,217839,227153,235577,324
8290,New Jersey -- Statewide,2010,3,229678,229789,229562,338
8291,New Jersey -- Statewide,2010,4,223643,221746,221561,345


In [23]:
##Prepare and save the 2010 dataset
# Calculate average monthly employment within each quarter.
# Require all three months; do not average incomplete quarters.
monthly_columns = [
    "month1_emplvl",
    "month2_emplvl",
    "month3_emplvl"
]

states_2010["avg_monthly_employment"] = (
    states_2010[monthly_columns].mean(axis=1, skipna=False)
)

# Retain identifiers, readable state labels, and outcome variables.
# Keep disclosure information for future checks.
selected_columns = [
    "area_fips",
    "area_title",
    "industry_code",
    "own_code",
    "year",
    "qtr",
    "disclosure_code",
    "avg_monthly_employment",
    "avg_wkly_wage"
]

# Organize records consistently by state and time.
prepared_2010 = (
    states_2010[selected_columns]
    .sort_values(["area_fips", "year", "qtr"])
    .reset_index(drop=True)
)

# Save the prepared CSV separately from the original ZIP archive.
prepared_path = (
    PROJECT_DIR / "data_processed"
    / "qcew_state_private_food_services_2010.csv"
)

# Avoid saving the DataFrame's row index as an extra CSV column.
prepared_2010.to_csv(prepared_path, index=False)

print(f"Saved: {prepared_path.name}")
print(f"Records: {len(prepared_2010)}")

# Display New Jersey's outcomes with readable number formatting.
# Formatting changes only the display, not the stored values.
display(
    prepared_2010.loc[
        prepared_2010["area_fips"].eq("34000")
    ].style.format({
        "avg_monthly_employment": "{:,.2f}",
        "avg_wkly_wage": "${:,.0f}"
    })
)

Saved: qcew_state_private_food_services_2010.csv
Records: 204


,area_fips,area_title,industry_code,own_code,year,qtr,disclosure_code,avg_monthly_employment,avg_wkly_wage
120,34000,New Jersey -- Statewide,722,5,2010,1,nan,"207,820.00",$313
121,34000,New Jersey -- Statewide,722,5,2010,2,nan,"226,856.33",$324
122,34000,New Jersey -- Statewide,722,5,2010,3,nan,"229,676.33",$338
123,34000,New Jersey -- Statewide,722,5,2010,4,nan,"222,316.67",$345


In [24]:
##Turn our preparation steps into a reusable function

def prepare_food_services(data, year):
    """Select and prepare state/DC private-sector food services data."""

    # Select the correct geography, ownership, industry, and year.
    selected = data.loc[
        data["area_fips"].isin(state_area_codes)
        & data["own_code"].eq(5)
        & data["agglvl_code"].eq(55)
        & data["industry_code"].eq("722")
        & data["year"].eq(year)
        & data["qtr"].isin([1, 2, 3, 4])
    ].copy()

    # Stop if a state appears more than once in the same quarter.
    if selected.duplicated(["area_fips", "year", "qtr"]).any():
        raise ValueError(f"{year}: duplicate state-quarter records.")

    # Confirm every expected state/DC is present in every quarter.
    expected = {
        (area, quarter)
        for area in state_area_codes
        for quarter in [1, 2, 3, 4]
    }
    actual = set(zip(selected["area_fips"], selected["qtr"]))

    if actual != expected:
        raise ValueError(f"{year}: incomplete state-quarter coverage.")

    # Pause for review if any disclosure flag is nonblank.
    flags = selected["disclosure_code"].fillna("").astype(str).str.strip()

    if flags.ne("").any():
        raise ValueError(f"{year}: disclosure flags require review.")

    # Require complete employment and wage values.
    monthly = ["month1_emplvl", "month2_emplvl", "month3_emplvl"]
    outcomes = monthly + ["avg_wkly_wage"]

    if selected[outcomes].isna().any().any():
        raise ValueError(f"{year}: missing outcome values.")

    # Calculate quarterly average monthly employment.
    selected["avg_monthly_employment"] = (
        selected[monthly].mean(axis=1, skipna=False)
    )

    # Return the same columns and ordering used in our saved 2010 file.
    return (
        selected[selected_columns]
        .sort_values(["area_fips", "year", "qtr"])
        .reset_index(drop=True)
    )


# Check that the function reproduces our already-prepared 2010 dataset.
check_2010 = prepare_food_services(food_2010, 2010)
pd.testing.assert_frame_equal(check_2010, prepared_2010)

print("Function verified: reproduces the prepared 2010 dataset.")

Function verified: reproduces the prepared 2010 dataset.


In [25]:
#Download and prepare 2011
# Set the year. The same workflow can later be reused for other years.
year = 2011

archive_url_2011 = (
    f"https://data.bls.gov/cew/data/files/"
    f"{year}/csv/{year}_qtrly_by_industry.zip"
)

archive_path_2011 = (
    PROJECT_DIR / "data_raw" / f"{year}_qtrly_by_industry.zip"
)

# Reuse a previously downloaded ZIP if available.
if archive_path_2011.exists() and is_zipfile(archive_path_2011):
    print(f"Using the saved {year} archive.")
else:
    # Download to a temporary file to distinguish incomplete downloads.
    temporary_path = archive_path_2011.with_suffix(".zip.part")

    print(f"Downloading the {year} archive...")

    with requests.get(
        archive_url_2011, stream=True, timeout=(30, 120)
    ) as response:
        response.raise_for_status()

        with temporary_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    # Rename only after confirming that the download is a ZIP.
    if not is_zipfile(temporary_path):
        raise ValueError("The download is not a valid ZIP archive.")

    temporary_path.replace(archive_path_2011)

# Find and read only the food services CSV inside the archive.
with ZipFile(archive_path_2011) as archive:
    matches = [
        name for name in archive.namelist()
        if " 722 " in name and name.endswith(".csv")
    ]

    if len(matches) != 1:
        raise ValueError("Expected exactly one industry 722 file.")

    with archive.open(matches[0]) as file:
        food_2011 = pd.read_csv(
            file,
            dtype={"area_fips": str, "industry_code": str}
        )

# Apply our verified filtering, validation, and calculation function.
prepared_2011 = prepare_food_services(food_2011, year)

# Save the prepared year separately from the raw archive.
output_path = (
    PROJECT_DIR / "data_processed"
    / f"qcew_state_private_food_services_{year}.csv"
)
prepared_2011.to_csv(output_path, index=False)

print(f"\nSaved: {output_path.name}")
print(f"Validated records: {len(prepared_2011)}")
print("\nRecords per quarter:")
print(prepared_2011.groupby("qtr").size())


Saved: qcew_state_private_food_services_2011.csv
Validated records: 204

Records per quarter:
qtr
1    51
2    51
3    51
4    51
dtype: int64


In [26]:
##Collect the remaining years, 2012–2018
# Collect each remaining year using the same preparation rules.
for year in range(2012, 2019):
    print(f"\nProcessing {year}...", flush=True)

    url = (
        f"https://data.bls.gov/cew/data/files/"
        f"{year}/csv/{year}_qtrly_by_industry.zip"
    )
    archive_file = (
        PROJECT_DIR / "data_raw" / f"{year}_qtrly_by_industry.zip"
    )

    # Skip downloading if a recognizable ZIP is already saved.
    if archive_file.exists() and is_zipfile(archive_file):
        print("Using saved archive.", flush=True)
    else:
        # Keep unfinished downloads separate from completed ZIP files.
        temporary_file = archive_file.with_suffix(".zip.part")

        with requests.get(
            url, stream=True, timeout=(30, 120)
        ) as response:
            response.raise_for_status()

            with temporary_file.open("wb") as file:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        file.write(chunk)

        if not is_zipfile(temporary_file):
            raise ValueError(f"{year}: download is not a valid ZIP.")

        temporary_file.replace(archive_file)

    # Read the food services CSV without extracting the whole archive.
    with ZipFile(archive_file) as archive:
        matches = [
            name for name in archive.namelist()
            if " 722 " in name and name.endswith(".csv")
        ]

        if len(matches) != 1:
            raise ValueError(f"{year}: expected one industry 722 file.")

        with archive.open(matches[0]) as file:
            yearly_data = pd.read_csv(
                file,
                dtype={"area_fips": str, "industry_code": str}
            )

    # Validate coverage, duplicates, disclosure flags, and missing values.
    # The function stops execution if a year needs investigation.
    prepared_year = prepare_food_services(yearly_data, year)

    # Save only after the year's checks pass.
    output_file = (
        PROJECT_DIR / "data_processed"
        / f"qcew_state_private_food_services_{year}.csv"
    )
    prepared_year.to_csv(output_file, index=False)

    print(
        f"{year}: saved {len(prepared_year)} validated records.",
        flush=True
    )

print("\nRemaining years completed successfully.")


Processing 2012...
2012: saved 204 validated records.

Processing 2013...
2013: saved 204 validated records.

Processing 2014...
2014: saved 204 validated records.

Processing 2015...
2015: saved 204 validated records.

Processing 2016...
2016: saved 204 validated records.

Processing 2017...
2017: saved 204 validated records.

Processing 2018...
2018: saved 204 validated records.

Remaining years completed successfully.


In [27]:
##Combine 2010–2018 into one panel dataset
# Read each prepared yearly CSV.
# Preserve geographic and industry codes as strings.
yearly_frames = []

for year in range(2010, 2019):
    file_path = (
        PROJECT_DIR / "data_processed"
        / f"qcew_state_private_food_services_{year}.csv"
    )

    frame = pd.read_csv(
        file_path,
        dtype={"area_fips": str, "industry_code": str}
    )
    yearly_frames.append(frame)

# Stack the yearly datasets vertically into one table.
panel = (
    pd.concat(yearly_frames, ignore_index=True)
    .sort_values(["area_fips", "year", "qtr"])
    .reset_index(drop=True)
)

# Verify that every expected state-year-quarter combination exists once.
# Expected size: 51 areas × 9 years × 4 quarters = 1,836 rows.
key_columns = ["area_fips", "year", "qtr"]

expected_keys = {
    (area, year, quarter)
    for area in state_area_codes
    for year in range(2010, 2019)
    for quarter in [1, 2, 3, 4]
}
actual_keys = set(
    panel[key_columns].itertuples(index=False, name=None)
)

if panel.duplicated(key_columns).any():
    raise ValueError("Duplicate state-year-quarter records found.")

if actual_keys != expected_keys:
    raise ValueError("Panel coverage does not match the expected study window.")

# Save the combined dataset for later exploration and analysis.
panel_path = (
    PROJECT_DIR / "data_processed"
    / "qcew_state_private_food_services_panel_2010_2018.csv"
)
panel.to_csv(panel_path, index=False)

print("Total records:", len(panel))
print("States/DC:", panel["area_fips"].nunique())
print("Years:", panel["year"].min(), "to", panel["year"].max())
print("\nRecords per year:")
print(panel.groupby("year").size())
print(f"\nSaved: {panel_path.name}")

Total records: 1836
States/DC: 51
Years: 2010 to 2018

Records per year:
year
2010    204
2011    204
2012    204
2013    204
2014    204
2015    204
2016    204
2017    204
2018    204
dtype: int64

Saved: qcew_state_private_food_services_panel_2010_2018.csv
